# Os Dutos do Q — Notebook 0 · Preparar o terreno

**Arquitetura de Dados · MBA FIAP · aula de Integração e Ingestão**

Na aula passada vocês construíram os órgãos do agente Q: conhecimento, memória e cache. O Q funcionou
porque alguém tinha colocado dado bom na frente dele. Hoje esse alguém são vocês.

Este notebook roda uma vez, no começo da aula, e faz três coisas: instala o que falta, traz o kit e
mostra o dado **como ele chegou**. Não ingira nada ainda. Olhar o dado sujo antes de tocar nele é a
primeira decisão de arquitetura da aula, e a única que não dá para terceirizar ao agente.

**Tempo:** 10 minutos. Se travar em alguma célula, chame o professor antes de seguir.

## Passo 1 — preparar a sessão

In [ ]:
# Passo 1 de 4 — preparar a sessão (roda uma vez, ~90 s)
!pip -q install deltalake duckdb sentence-transformers pyyaml pyarrow pandas 2>&1 | tail -1

import os, sys, json, shutil
from pathlib import Path

# O kit vem de um zip. Troque KIT_URL pelo endereço que o professor passar,
# ou faça upload de dutos-do-q.zip no painel de arquivos do Colab (ícone de pasta à esquerda).
KIT_URL = os.environ.get("KIT_URL", "")
RAIZ = Path("/content") if Path("/content").exists() else Path.cwd()
KIT = RAIZ / "dutos-do-q"

if not KIT.exists():
    zip_local = RAIZ / "dutos-do-q.zip"
    if KIT_URL and not zip_local.exists():
        !wget -q -O {zip_local} {KIT_URL}
    assert zip_local.exists(), "Faça upload de dutos-do-q.zip no painel de arquivos, ou preencha KIT_URL."
    shutil.unpack_archive(str(zip_local), str(RAIZ))

sys.path.insert(0, str(KIT / "kit"))
os.chdir(KIT)
print("kit em", KIT)
print(sorted(p.name for p in (KIT / "kit").iterdir()))

## Passo 2 — o que a Quantum depositou no inbox

Quatro fontes, formatos diferentes, qualidade diferente. Repare no que cada uma é: um CSV de clientes,
um CSV de transações, um Parquet de tarifas e uma pasta de documentos em Markdown com front-matter.
Nenhuma delas veio com schema declarado, e nenhuma delas prometeu estar correta.

In [ ]:
INBOX = KIT / "dados" / "inbox"
for p in sorted(INBOX.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(INBOX)!s:<45} {p.stat().st_size/1024:>8.1f} KB")

## Passo 3 — olhe o dado sujo antes de escrever qualquer regra

As células abaixo mostram as armadilhas plantadas. Cada uma delas vai virar uma linha do contrato que
vocês escrevem no próximo notebook. Anotem: **quantas vocês conseguem nomear antes de olhar a resposta?**

In [ ]:
import pandas as pd
clientes = pd.read_csv(INBOX / "clientes.csv", dtype=str)
print("clientes:", len(clientes), "linhas")
print(clientes[clientes.cliente_id.isin(["C0021","C0022","C0041","C0031","C0032"])].to_string(index=False))

In [ ]:
transacoes = pd.read_csv(INBOX / "transacoes.csv", dtype=str)
print("transacoes:", len(transacoes), "linhas")
print(transacoes.tail(8).to_string(index=False))

In [ ]:
tarifas = pd.read_parquet(INBOX / "tarifas.parquet")
print(tarifas.to_string(index=False))

In [ ]:
# documentos: 25 arquivos, 21 doc_id distintos. Quatro deles têm duas versões.
docs = sorted((INBOX / "docs").glob("*.md"))
print(len(docs), "arquivos")
print((INBOX / "docs" / "prod-tabela-tarifas__v1.md").read_text()[:400])

### As armadilhas, agora sem suspense

| Fonte | O que está errado | Quantas |
|---|---|---|
| clientes | CPF que não fecha no dígito verificador | 2 |
| clientes | segmento fora do domínio (`vip`) | 1 |
| clientes | mesma pessoa duas vezes, com espaços em volta do nome | 2 |
| transações | data que não existe (30/02, mês 13) | 2 |
| transações | depósito com valor negativo | 2 |
| transações | cliente que não existe no cadastro | 2 |
| transações | tipo fora do domínio, valor ausente | 2 |
| tarifas | duas vigências de saque se sobrepondo | 1 |
| documentos | quatro documentos com v1 e v2 no mesmo lote | 4 pares |
| documentos | três de tipo não autoritativo (marketing, rascunho, faq antigo) | 3 |

Doze linhas erradas mais duas duplicatas. É pouco, e é de propósito: o problema da aula não é volume,
é decidir o que fazer com cada uma. Rejeitar, corrigir ou deixar passar são três arquiteturas diferentes.

## Passo 4 — criar o lakehouse vazio

Delta Lake de verdade, numa pasta. Sem cluster, sem conta, sem token. As tabelas nascem vazias e com o
schema declarado: isso não é burocracia, é o que impede uma coluna de mudar de tipo entre duas execuções
e derrubar o duto no meio da aula.

In [ ]:
from lake import Lake
lake = Lake(str(KIT / "lakehouse"))
lake.zerar().criar_todas()
print(lake.resumo().to_string(index=False))

As tabelas com `cdf = True` têm **Change Data Feed** ligado. Elas guardam não só o estado atual, mas o
registro de cada linha que entrou, mudou ou saiu. É o que o Bloco 2 vai consumir. As outras não precisam,
e ligar CDF em tudo custa espaço sem dar nada em troca.

Fim do notebook 0. Abra o **01_bloco1_batch.ipynb**.